In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier

In [2]:
df = pd.read_csv("../data/features.csv")
X = df.drop(columns=["mod", "snr"])
y = df["mod"]
snrs = df["snr"]
print(X.shape)

(220000, 12)


In [3]:
strata = y + "_" + snrs.astype(str)

X_train, X_test, y_train, y_test, snr_train, snr_test = train_test_split(
    X, y, snrs,
    test_size=0.2,
    stratify=strata,
    random_state=42,
)

In [4]:
strata_train = y_train + "_" + snr_train.astype(str)

X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train,
    train_size=20000,
    stratify=strata_train,
    random_state=42,
)
print(X_tune.shape)

(20000, 12)


In [5]:
baseline = DummyClassifier(strategy="most_frequent")
baseline_scores = cross_val_score(baseline, X_tune, y_tune, cv=5)
print(f"Baseline: {baseline_scores.mean():.3f} ± {baseline_scores.std():.3f}")

Baseline: 0.091 ± 0.000


In [6]:
models = {
    "k-NN": (
        KNeighborsClassifier(),
        {"kneighborsclassifier__n_neighbors": [5, 15, 31, 61]},
    ),
}

In [7]:
results = {}

for name, (model, param_grid) in models.items():
    print(f"\nTraining {name}")
    pipeline = make_pipeline(StandardScaler(), model)

    search = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
    search.fit(X_tune, y_tune)

    best = search.best_index_
    results[name] = {
        "mean_accuracy": search.best_score_,
        "std": search.cv_results_["std_test_score"][best],
        "best_params": search.best_params_,
    }
    print(f"{search.best_score_:.3f} ± {results[name]['std']:.3f}  {search.best_params_}")


Training k-NN
0.418 ± 0.003  {'kneighborsclassifier__n_neighbors': 31}


In [8]:
results_df = pd.DataFrame(results).T.sort_values("mean_accuracy", ascending=False)
results_df.to_csv("../figures/cv_results.csv")
results_df

,mean_accuracy,std,best_params
k-NN,0.41755,0.002865,{'kneighborsclassifier__n_neighbors': 31}
